## defines

In [6]:
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel
from pymongo.errors import OperationFailure
from pprint import pprint
from dotenv import load_dotenv
import os, time
from langchain_core.tools import tool
from typing import List
from typing import Annotated
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import ToolMessage
from langgraph.graph import END, StateGraph, START
import voyageai


load_dotenv()
print("MONGO_URI loaded:", bool(os.getenv("MONGO_URI")))
print("GOOGLE_API_KEY loaded:", bool(os.getenv("GOOGLE_API_KEY")))
print("VOYAGEAI_API_KEY loaded:", bool(os.getenv("VOYAGEAI_API_KEY")))

MONGO_URI loaded: True
GOOGLE_API_KEY loaded: True
VOYAGEAI_API_KEY loaded: True


### connect to mongo DB

In [7]:
def init_mongodb():
    """
    Initialize MongoDB client and collections.

    Returns:
        tuple: MongoDB client, vector search collection, full documents collection.
    """
    mongodb_client = MongoClient(os.getenv("MONGO_URI"))
    
    DB_NAME = "ai_agents"
    
    vs_collection = mongodb_client[DB_NAME]["chunked_docs"]
    
    full_collection = mongodb_client[DB_NAME]["full_docs"]
    
    return mongodb_client, vs_collection, full_collection


### graph

In [8]:
# Define the graph state type with messages that can accumulate
class GraphState(TypedDict):
    # Define a messages field that keeps track of conversation history
    messages: Annotated[list, add_messages]

### generate embedding

In [9]:
def generate_embedding(text: str) -> List[float]:
    """
    Generate embedding for a piece of text.

    Args:
        text (str): The text to embed.
        embedding_model (voyage-4): The embedding model.

    Returns:
        List[float]: The embedding of the text.
    """

    embedding_model = voyageai.Client(api_key=os.getenv("VOYAGEAI_API_KEY"))

    embedding = embedding_model.embed(text, model="voyage-4", input_type="query").embeddings[0]
    
    return embedding


## tool: get_information_for_question_answering

In [10]:
@tool 
def get_information_for_question_answering(user_query: str) -> str:
    """
    Retrieve relevant documents for a user query using vector search.

    Args:
        user_query (str): The user's query.

    Returns:
        str: The retrieved documents as a string.
    """

    query_embedding = generate_embedding(user_query)

    vs_collection = init_mongodb()[1]
    
    pipeline = [
        {
            # Use vector search to find similar documents
            "$vectorSearch": {
                "index": "vector_index",  # Name of the vector index
                "path": "embedding",       # Field containing the embeddings
                "queryVector": query_embedding,  # The query embedding to compare against
                "numCandidates": 150,      # Consider 150 candidates (wider search)
                "limit": 5,                # Return only top 5 matches
            }
        },
        {
            # Project only the fields we need
            "$project": {
                "_id": 0,                  # Exclude document ID
                "body": 1,                 # Include the document body
                "score": {"$meta": "vectorSearchScore"},  # Include the similarity score
            }
        },
    ]
    
    results = vs_collection.aggregate(pipeline)
    
    context = "\n\n".join([doc.get("body") for doc in results])
    
    return context


## tool: get_page_content_for_summarization

In [11]:
@tool 
def get_page_content_for_summarization(user_query: str) -> str:
    """
    Retrieve the content of a documentation page for summarization.

    Args:
        user_query (str): The user's query (title of the documentation page).

    Returns:
        str: The content of the documentation page.
    """
    full_collection = init_mongodb()[2]

    query = {"title": user_query}
    
    projection = {"_id": 0, "body": 1}
    
    document = full_collection.find_one(query, projection)
    
    if document:
        return document["body"]
    else:
        return "Document not found"

### agent node

In [12]:
def agent(state: GraphState, llm_with_tools) -> GraphState:
    """
    Agent node.

    Args:
        state (GraphState): The graph state.
        llm_with_tools: The LLM with tools.

    Returns:
        GraphState: The updated messages.
    """

    messages = state["messages"]
    
    result = llm_with_tools.invoke(messages)
    
    return {"messages": [result]}

### Tool node

In [13]:
def tool_node(state: GraphState, tools_by_name) -> GraphState:
    """
    Tool node.

    Args:
        state (GraphState): The graph state.
        tools_by_name (Dict[str, Callable]): The tools by name.

    Returns:
        GraphState: The updated messages.
    """
    result = []
    
    tool_calls = state["messages"][-1].tool_calls
    
    for tool_call in tool_calls:
        tool = tools_by_name[tool_call["name"]]
        
        observation = tool.invoke(tool_call["args"])
        
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    
    return {"messages": result}

### route tools

In [14]:
def route_tools(state: GraphState):
    """
    Route to the tool node if the last message has tool calls. Otherwise, route to the end.

    Args:
        state (GraphState): The graph state.

    Returns:
        str: The next node to route to.
    """
    messages = state.get("messages", [])
    
    if len(messages) > 0:
        ai_message = messages[-1]
    else:
        raise ValueError(f"No messages found in input state to tool_edge: {state}")
    
    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        return "tools"
    
    return END


## Initialize the graph

In [15]:
def init_graph(llm_with_tools, tools_by_name):
    """
    Initialize the graph.

    Args:
        llm_with_tools: The LLM with tools.
        tools_by_name (Dict[str, Callable]): The tools by name.
        mongodb_client (MongoClient): The MongoDB client.

    Returns:
        StateGraph: The compiled graph.
    """
    graph = StateGraph(GraphState)
    
    graph.add_node("agent", lambda state: agent(state, llm_with_tools))
    
    graph.add_node("tools", lambda state: tool_node(state, tools_by_name))
    
    graph.add_edge(START, "agent")
    
    graph.add_edge("tools", "agent")
    
    graph.add_conditional_edges("agent", route_tools, {"tools": "tools", END: END})
    
    return graph.compile()

### execute the graph

In [16]:
def execute_graph(app, user_input: str) -> None:
    """
    Stream outputs from the graph.

    Args:
        app: The compiled graph application.
        thread_id (str): The thread ID.
        user_input (str): The user's input.
    """
    input = {"messages": [("user", user_input)]}

    
    for output in app.stream(input):
        for key, value in output.items():
            print(f"Node {key}:")
            print(value)
    
    print("---FINAL ANSWER---")
    
    print(value["messages"][-1].content)

## our main function

In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI

def main():
    """
    Main function to initialize and execute the graph.
    """
    mongodb_client, vs_collection, full_collection = init_mongodb()
    
    tools = [
        get_information_for_question_answering,
        get_page_content_for_summarization
    ]
    
    llm = ChatGoogleGenerativeAI(
        model="gemini-3.6-flash",
        temperature=0.1,
        api_key=os.getenv("GOOGLE_API_KEY")
    )    
    
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "You are a helpful AI assistant."
                " You are provided with tools to answer questions and summarize technical documentation related to MongoDB."
                " Think step-by-step and use these tools to get the information required to answer the user query."
                " Do not re-run tools unless absolutely necessary."
                " If you are not able to get enough information using the tools, reply with I DON'T KNOW."
                " You have access to the following tools: {tool_names}."
            ),
            MessagesPlaceholder(variable_name="messages"),
        ]
    )
    
    prompt = prompt.partial(tool_names=", ".join([tool.name for tool in tools]))
    
    bind_tools = llm.bind_tools(tools)
    
    llm_with_tools = prompt | bind_tools
    
    tools_by_name = {tool.name: tool for tool in tools}
    
    app = init_graph(llm_with_tools, tools_by_name)
    
    execute_graph(app, "What are some best practices for data backups in MongoDB?")
    
    execute_graph(app, "Give me a summary of the page titled Create a MongoDB Deployment")

# invoke main

In [18]:
main()

C:\Users\yarde\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Node agent:
{'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_information_for_question_answering', 'arguments': '{"user_query": "best practices for data backups in MongoDB"}'}, '__gemini_function_call_thought_signatures__': {'call_562669': 'EtADCs0DAWkUfRMDkJY0YqQNxtnbZsrwY/HuBnhfDJ1n9HaglZd1xg2rMgqSuzSMvX2KzT5usQbr0OFQxwU4VHXkixy1Jl1z6SdEz77Ii1/K0h8sQMLZkr5LOme39TI0Ov84Uvnch20urdkv/rc1oHIBZ7h/Nrkt0o8TeqmH5BCnW54wSQzNCH1lw+0Ih8It5im/NqGNrNYB5XBp5Z1uYkBwNNXkHdx0rfsT90jIiSGCCIfHdyi2oYOPAgCcK67JMTPaNcofnwOE5WK9iL8Yhu+V7yb/QBmnlBsXkXFQC+PVlOzsjN6umdzinutTqaJcy9XmA73SyGuUpnRqs3xYbsLQIcRlWj//zbMkMZCapv1Bvg0PCjOnrk2Z0gXWNIwRX9FyAbDHeC1fdXXJ30cVhRNggknwgs8iXHtYMr187ecaT5e7SOgPWsHCfs1s4uqWL9r58omhJr3TiloAMbdZnmIEDIZcKW5mzVHgHWFPCZzDI30ywVyoROdzar6Ge11XelHln8exMPG0lwq8S+q/Vi71v1oUM2rm/Es4vCkUf4J5Ou+BDqjoaHGVXy0ahnmTdTEDSHiabIu5OMAvZj80dWlMGYKdYUxqm/yTPlG0tUrvots='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings'

C:\Users\yarde\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Node agent:
{'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_information_for_question_answering', 'arguments': '{"user_query": "MongoDB backup strategies best practices mongodump filesystem snapshot Atlas Ops Manager"}'}, '__gemini_function_call_thought_signatures__': {'call_65407': 'ErsBCrgBAWkUfRNHoKomga84rqymBr76xpHELBEXG/Tx0PX6YZTioeD0Td1iFYnfqL1zfdbFwcxto3PW3waLNsTOjl1PdOG1khZRnv9W3bTSqriUlFABVdrXfrRJujgtur/k6W6SDtx9o2I25DrUNGMQSTkVe2jloBrklBct2On5WlpxaXbnztZfxedsHA03uPfB4qArSrTAhSI8IwFAoaXYtXRiSH8bHDmq3jmoIXaSZzV1G1+wqYKATsYoRA=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1a5c7-9704-7061-ad8e-ef672bfb3408-0', tool_calls=[{'name': 'get_information_for_question_answering', 'args': {'user_query': 'MongoDB backup strategies best practices mongodump filesystem snapshot Atlas Ops Manager'}, 'id': 'call_65407', 'type': 'tool_call'}], invalid

C:\Users\yarde\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Node agent:
{'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_page_content_for_summarization', 'arguments': '{"user_query": "MongoDB Backup Methods"}'}, '__gemini_function_call_thought_signatures__': {'call_37519': 'Eu8BCuwBAWkUfRN4Y+r7EKj++MP+doe7Kshhh3YkJfFzA4K5G/l3ev7zyIse7ffzFmd7FbBnCbSoixC4TBP36assI47GuIrhGGQVVwQJ5OPPZE5UcjW4jFVMtreqVvRKt40i/qyJVorFvHbDxx0zjaKYlBNseUguqrxb8dhNIDKQk+VePL8a8ciLgtlC3O9LyHHx9XFtUnlugSzMRH9dSRt5BGHtPrGqClhF9FEMvvVwE8FZSZuDr1EOlb6SNnbxiiylUMq+3pBCc+hJNbbZoOiy1UPMsmmILHTiAVbd3UA2AE7WpAP6QHpo6mS98CFXUds='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1a5c7-a2b7-7b62-9e37-33e3a96e3a52-0', tool_calls=[{'name': 'get_page_content_for_summarization', 'args': {'user_query': 'MongoDB Backup Methods'}, 'id': 'call_37519', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1661, 'output_tokens': 5

... (4 outputs left out)

C:\Users\yarde\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Node agent:
{'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_page_content_for_summarization', 'arguments': '{"user_query": "Create a MongoDB Deployment"}'}, '__gemini_function_call_thought_signatures__': {'call_439464': 'EoIDCv8CAWkUfROj8YbDYVUmjhOu2t83689xNZD+ESokNueNf4VL8cw5uNqMmQ4cpR3s0ESNOqLrcyMs0cubAeg+YEwAVQRgcryO88AD+tpPLSiV0qOIiIHVfEmK/s++O1vUJXGj2gdMyVKoWCViUYZO66KvkDpg0wcdCCkR/OwI2TebxgkDhtbKBXd3vGaiYiwnsFSQfiY5Sqw0mSWSFwzWCCgKGBJr14bpQYPwad65DInUYFfRFGe/lz7e8NrmqpTKP2dejPTYPyg70WrPF6UW49/seB16KHL2DiHAPpcFi7uScx1S6/7RpAbJurZUByBP0vx9rt6aALZGJuCS7MVJQAl10FFX2uY2YCNI7PtE4OY+bNzLuj7NN1fa3sX8uznC1z/bjJAU0ldOQF5Z6OXlcg8DlGJ4CsJj5UkWYGBTIHsCGi6t3tDlbc3pCogYHCiFxsHkteR1DaPebqAZ1+gZfdY8W1mw1k5unhWhpxIE+4lxQb+kDAIoSiDXmWQ2NGl8lVQ='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1a5c7-d5d6-7bc0-adba-6ba21d3ea574-0', tool_calls=[{'name': 'get_pag

C:\Users\yarde\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash, description: Generate content requests per minute limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 583.367512ms.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Gene